In [ ]:
import yfinance as yf
import pandas as pd
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

def run_spx_0dte_final():
    ticker = "^SPX"
    print(f"--- SPX 0DTE FINAL EXECUTION SCAN (V13 Logic) ---")
    
    # 1. Fetch Data
    df = yf.download(ticker, period="5d", interval="1d", auto_adjust=True)
    
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0 if 'Open' in df.columns.get_level_values(0) else 1)
    df.columns = [str(col).capitalize() for col in df.columns]

    # 2. Candles for Logic
    c = df.iloc[-1]      # Current Market Day (Target)
    c1 = df.iloc[-2]     # Previous Market Day
    
    # 3. TECHNICAL LOGIC (V13: Gap + 2 Wicks)
    # 100% Confluence of these 3 items is required.
    b_score, s_score = 0, 0
    MIN_GAP_PCT = 0.0001 # 0.01% Any Gap

    # A. Gap (Criteria 1)
    gap_pct = (c['Open'] - c1['Close']) / c1['Close']
    if gap_pct >= MIN_GAP_PCT: b_score += 33.3
    elif gap_pct <= -MIN_GAP_PCT: s_score += 33.3

    # B. Upper Wick (Criteria 2)
    c1_open, c1_close, c1_high, c1_low = c1['Open'], c1['Close'], c1['High'], c1['Low']
    c1_body = abs(c1_open - c1_close)
    c1_is_green = c1_close > c1_open
    u_wick = c1_high - max(c1_open, c1_close)
    
    if u_wick < c1_body: # Continuation
        if c1_is_green: b_score += 33.3
        else: s_score += 33.3
    else: # Reversal
        if c1_is_green: s_score += 33.3
        else: b_score += 33.3

    # C. Lower Wick (Criteria 3)
    l_wick = min(c1_open, c1_close) - c1_low
    if l_wick < c1_body: # Continuation
        if c1_is_green: b_score += 33.3
        else: s_score += 33.3
    else: # Reversal
        if c1_is_green: s_score += 33.3
        else: b_score += 33.3

    # --- FINAL SIGNAL ---
    direction = "NEUTRAL"
    if b_score > 99: direction = "BULLISH"
    elif s_score > 99: direction = "BEARISH"

    print("\n" + "="*60)
    print(f"RESULT: {direction}")
    print("="*60)
    print(f"Technical Confluence: {'100%' if direction != 'NEUTRAL' else '0%'}")
    print(f"Opening Gap:          {gap_pct*100:.4f}%")
    print("-" * 60)

    if direction != "NEUTRAL":
        print("COPY PROMPT FOR GOOGLE AI:")
        print("-" * 40)
        print(f"MISSION: You are an institutional macro trader. Analyze the S&P 500 (^SPX) for a 0 DTE holding period.")
        print(f"TECHNICAL SIGNAL: {direction}")
        print(f"CONTEXT: The technicals show a high-probability candle confluence in this direction.")
        print(f"REQUEST: Check today's news (CPI, Fed, Earnings) and pre-market sentiment.")
        print(f"OUTPUT: Provide a final recommendation: 'CONFIRM {direction}' or 'REJECT'.")
        print(f"Note: Trade only if AI result is 'CONFIRM'.")
        print("-" * 40)
    else:
        print("No signal today based on Wick/Gap confluence.")

if __name__ == "__main__":
    run_spx_0dte_final()